# Lab Problem Statement 6: Customer Segmentation and Predictive Analytics Using Machine Learning

| | |
|---|---|
| **Name** | Tanisha Sharma |
| **Roll No.** | 23102C0077 |
| **Platform** | Python, Google Colab |
| **Git Repository** | *(paste your repository link here before submitting)* |

---

## 1. Problem Statement
An e-commerce company wants to better understand its customers and improve its marketing decisions. Using historical online retail transaction data, this notebook develops a machine-learning-based analytics solution that:

1. identifies meaningful **customer segments** (K-Means and Hierarchical Clustering), and
2. **predicts** whether a customer belongs to the high-value customer segment (Random Forest and SVM).

## 2. Dataset Description
**UCI Machine Learning Repository: Online Retail Dataset** (541,909 transactions from a UK-based online retailer, 01/12/2010 to 09/12/2011).

| Attribute | Description |
|---|---|
| InvoiceNo | Invoice number. A leading `C` marks a cancellation |
| StockCode | Product (item) code |
| Description | Product name |
| Quantity | Units per transaction |
| InvoiceDate | Date and time of the invoice |
| UnitPrice | Price per unit (GBP) |
| CustomerID | Unique customer identifier |
| Country | Customer's country |

## 3. Workflow
Preprocessing, then customer-level feature engineering, then outlier treatment and scaling, then Elbow Method and K-Means, then Hierarchical Clustering and dendrogram, then Silhouette comparison, then PCA visualisation, then cluster profiling, then Random Forest and SVM prediction, then feature importance, then interactive 3D Plotly view, then marketing recommendations.

## 4. Setup and Imports

In [ ]:
import warnings, os, zipfile, urllib.request
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance
from sklearn.metrics import (silhouette_score, adjusted_rand_score, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)
from scipy.cluster.hierarchy import linkage, dendrogram

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42
STUDENT_NAME = "Tanisha Sharma"
ROLL_NO = "23102C0077"
print(f"Student: {STUDENT_NAME} | Roll No: {ROLL_NO}")

## 5. Load the Dataset
The data is downloaded directly from the UCI repository. Reading the Excel file takes about 1 to 2 minutes. If the download fails, the cell asks you to upload `Online Retail.xlsx` manually.

In [ ]:
URL = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
DATA_DIR = "online_retail_data"
os.makedirs(DATA_DIR, exist_ok=True)

def find_xlsx():
    for f in os.listdir(DATA_DIR):
        if f.lower().endswith(".xlsx"):
            return os.path.join(DATA_DIR, f)
    return None

xlsx_path = find_xlsx()
if xlsx_path is None:
    try:
        req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
        zpath = os.path.join(DATA_DIR, "online_retail.zip")
        with urllib.request.urlopen(req, timeout=120) as r, open(zpath, "wb") as f:
            f.write(r.read())
        with zipfile.ZipFile(zpath) as z:
            z.extractall(DATA_DIR)
        xlsx_path = find_xlsx()
    except Exception as e:
        print("Automatic download failed:", e)
        print("Please upload 'Online Retail.xlsx' now.")
        from google.colab import files
        up = files.upload()
        xlsx_path = list(up.keys())[0]

df_raw = pd.read_excel(xlsx_path, engine="openpyxl")
print("Loaded:", xlsx_path)
print("Shape:", df_raw.shape)
df_raw.head()

## 6. Exploratory Data Analysis and Data Quality Check

In [ ]:
print(df_raw.dtypes, "\n")
miss = df_raw.isna().sum().to_frame("Missing")
miss["Missing %"] = (miss["Missing"] / len(df_raw) * 100).round(2)
display(miss)

inv = df_raw["InvoiceNo"].astype(str)
print(f"\nCancellation lines (InvoiceNo starts with 'C'): {inv.str.startswith('C').sum():,}")
print(f"Rows with Quantity <= 0 : {(df_raw['Quantity'] <= 0).sum():,}")
print(f"Rows with UnitPrice <= 0: {(df_raw['UnitPrice'] <= 0).sum():,}")
print(f"Duplicate rows          : {df_raw.duplicated().sum():,}")
display(df_raw[["Quantity", "UnitPrice"]].describe().round(2))

fig, ax = plt.subplots(figsize=(8, 4))
df_raw["Country"].value_counts().head(10).plot(kind="barh", ax=ax, color="steelblue")
ax.invert_yaxis(); ax.set_title("Top 10 Countries by Number of Transaction Lines"); ax.set_xlabel("Rows")
plt.tight_layout(); plt.show()

## 7. Data Preprocessing
Cleaning decisions:

| Issue | Treatment | Reason |
|---|---|---|
| Missing `CustomerID` | Rows dropped | Segmentation is at customer level, so anonymous rows cannot be assigned |
| Cancellations (`C...` invoices) | Removed | They carry negative quantities and would distort spend |
| Quantity <= 0 or UnitPrice <= 0 | Removed | Adjustments, damaged stock, data-entry errors |
| Exact duplicate rows | Removed | Double-counted lines inflate frequency and spend |

In [ ]:
df = df_raw.copy()
df.columns = [c.strip() for c in df.columns]
df["InvoiceNo"] = df["InvoiceNo"].astype(str)

steps = [("Raw rows", len(df))]

df = df.dropna(subset=["CustomerID"]);                       steps.append(("After dropping missing CustomerID", len(df)))
df = df[~df["InvoiceNo"].str.startswith("C")];               steps.append(("After removing cancellations", len(df)))
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)];       steps.append(("After removing Qty<=0 / Price<=0", len(df)))
df = df.drop_duplicates();                                   steps.append(("After removing duplicates", len(df)))

df["CustomerID"] = df["CustomerID"].astype(int)
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

summary = pd.DataFrame(steps, columns=["Step", "Rows"])
summary["Dropped"] = -summary["Rows"].diff().fillna(0).astype(int)
display(summary)
print("Unique customers:", df["CustomerID"].nunique())
print("Period:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())

## 8. Customer-Level Feature Engineering
| Feature | Definition |
|---|---|
| **Recency** | Days since the customer's last purchase (snapshot = last date + 1 day) |
| **Frequency** | Number of distinct invoices |
| **Monetary** | Total spend |
| **AvgTransactionValue** | Monetary / Frequency |
| **TotalQuantity** | Total units purchased |
| **AvgQuantityPerInvoice** | TotalQuantity / Frequency |
| **UniqueProducts** | Number of distinct products bought |
| **PurchaseFrequency** | Invoices per month since the customer's first purchase |

In [ ]:
snapshot = df["InvoiceDate"].max() + pd.Timedelta(days=1)
g = df.groupby("CustomerID")

cust = pd.DataFrame({
    "Recency": (snapshot - g["InvoiceDate"].max()).dt.days,
    "Frequency": g["InvoiceNo"].nunique(),
    "Monetary": g["TotalPrice"].sum(),
    "TotalQuantity": g["Quantity"].sum(),
    "UniqueProducts": g["StockCode"].nunique(),
})
cust["AvgTransactionValue"] = cust["Monetary"] / cust["Frequency"]
cust["AvgQuantityPerInvoice"] = cust["TotalQuantity"] / cust["Frequency"]
lifetime_months = ((snapshot - g["InvoiceDate"].min()).dt.days.clip(lower=1)) / 30.44
cust["PurchaseFrequency"] = cust["Frequency"] / lifetime_months

# Features used for clustering (UniqueProducts and AvgQuantityPerInvoice are kept for profiling only)
CLUSTER_FEATURES = ["Recency", "Frequency", "Monetary", "AvgTransactionValue", "TotalQuantity", "PurchaseFrequency"]
PROFILE_FEATURES = CLUSTER_FEATURES + ["AvgQuantityPerInvoice", "UniqueProducts"]

print("Customer table shape:", cust.shape)
display(cust.describe().round(2).T)
cust.head()

### 8.1 Distributions and Correlations (before treatment)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), CLUSTER_FEATURES):
    sns.histplot(cust[col], bins=50, kde=False, ax=ax, color="tomato")
    ax.set_title(f"{col} (skew = {cust[col].skew():.1f})")
plt.suptitle("Raw feature distributions: heavily right-skewed", y=1.02)
plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 5.5))
sns.heatmap(cust[CLUSTER_FEATURES].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation between clustering features"); plt.tight_layout(); plt.show()

## 9. Outlier Treatment and Feature Scaling
All features are strongly right-skewed (a few very large buyers). Two steps are applied:

1. **`log1p` transform** to compress the long tail.
2. **Winsorisation** at the 1st and 99th percentiles to cap remaining extreme values (customers are kept, not deleted).

After that, **`StandardScaler`** gives every feature zero mean and unit variance, which is required because K-Means and SVM are distance-based.

In [ ]:
X_log = np.log1p(cust[CLUSTER_FEATURES])

X_proc = X_log.copy()
for col in CLUSTER_FEATURES:
    lo, hi = np.percentile(X_proc[col], [1, 99])
    X_proc[col] = X_proc[col].clip(lo, hi)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_proc)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), CLUSTER_FEATURES):
    sns.histplot(X_proc[col], bins=40, ax=ax, color="seagreen")
    ax.set_title(f"log1p({col}) after capping (skew = {X_proc[col].skew():.2f})")
plt.suptitle("Distributions after log transform and winsorisation", y=1.02)
plt.tight_layout(); plt.show()

print("Skewness before:\n", cust[CLUSTER_FEATURES].skew().round(2).to_dict())
print("Skewness after :\n", X_proc.skew().round(2).to_dict())
print("Scaled matrix shape:", X_scaled.shape)

## 10. Optimal Number of Clusters: Elbow Method (with Silhouette support)

In [ ]:
K_RANGE = range(2, 11)
inertias, sil_scores = [], []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(list(K_RANGE), inertias, "o-", color="navy")
axes[0].set_title("Elbow Method: Inertia (WCSS) vs k"); axes[0].set_xlabel("Number of clusters (k)"); axes[0].set_ylabel("Inertia")
axes[1].plot(list(K_RANGE), sil_scores, "s-", color="darkorange")
axes[1].set_title("Average Silhouette Score vs k"); axes[1].set_xlabel("Number of clusters (k)"); axes[1].set_ylabel("Silhouette")
plt.tight_layout(); plt.show()

# k=2 usually wins on silhouette alone but is too coarse for marketing; choose the best silhouette among k = 3..6
cands = {k: s for k, s in zip(K_RANGE, sil_scores) if 3 <= k <= 6}
K = max(cands, key=cands.get)
display(pd.DataFrame({"k": list(K_RANGE), "Inertia": np.round(inertias, 1), "Silhouette": np.round(sil_scores, 4)}))
print(f"Chosen k = {K} (elbow region plus highest silhouette among 3..6 = {cands[K]:.4f})")

## 11. K-Means Customer Segmentation

In [ ]:
kmeans = KMeans(n_clusters=K, n_init=20, random_state=RANDOM_STATE).fit(X_scaled)
cust["Cluster"] = kmeans.labels_
sil_km = silhouette_score(X_scaled, kmeans.labels_)

print(f"K-Means (k={K}) Silhouette Score: {sil_km:.4f}")
print(cust["Cluster"].value_counts().sort_index().to_string())

## 12. Hierarchical Clustering and Dendrogram
Ward linkage (minimises within-cluster variance) is used on the same scaled data. The dendrogram is truncated to the last 30 merges for readability.

In [ ]:
Z = linkage(X_scaled, method="ward")

plt.figure(figsize=(12, 5.5))
dendrogram(Z, truncate_mode="lastp", p=30, leaf_rotation=90, leaf_font_size=9, show_contracted=True)
cut_height = Z[-(K - 1), 2] - 1e-6
plt.axhline(cut_height, color="red", ls="--", label=f"Cut for k = {K}")
plt.title("Hierarchical Clustering Dendrogram (Ward linkage, truncated)")
plt.xlabel("Cluster (size in brackets where merged)"); plt.ylabel("Ward distance")
plt.legend(); plt.tight_layout(); plt.show()

hier = AgglomerativeClustering(n_clusters=K, linkage="ward").fit(X_scaled)
cust["HCluster"] = hier.labels_
sil_hc = silhouette_score(X_scaled, hier.labels_)
print(f"Hierarchical (k={K}) Silhouette Score: {sil_hc:.4f}")

## 13. Silhouette Score and Comparison of Clustering Results

In [ ]:
ari = adjusted_rand_score(cust["Cluster"], cust["HCluster"])
comparison = pd.DataFrame({
    "Method": ["K-Means", "Hierarchical (Ward)"],
    "k": [K, K],
    "Silhouette Score": [round(sil_km, 4), round(sil_hc, 4)],
    "Smallest cluster": [cust["Cluster"].value_counts().min(), cust["HCluster"].value_counts().min()],
    "Largest cluster": [cust["Cluster"].value_counts().max(), cust["HCluster"].value_counts().max()],
})
display(comparison)
print(f"Adjusted Rand Index between the two partitions: {ari:.4f} (1.0 = identical)")
print("\nCross-tabulation (rows = K-Means, columns = Hierarchical):")
display(pd.crosstab(cust["Cluster"], cust["HCluster"]))

# Per-sample silhouette plot for K-Means
from sklearn.metrics import silhouette_samples
ss = silhouette_samples(X_scaled, kmeans.labels_)
fig, ax = plt.subplots(figsize=(8, 5))
y_lo = 10
for c in range(K):
    vals = np.sort(ss[kmeans.labels_ == c])
    ax.fill_betweenx(np.arange(y_lo, y_lo + len(vals)), 0, vals, alpha=0.7)
    ax.text(-0.05, y_lo + 0.5 * len(vals), str(c))
    y_lo += len(vals) + 10
ax.axvline(sil_km, color="red", ls="--", label=f"Mean = {sil_km:.3f}")
ax.set_title("K-Means Silhouette Plot"); ax.set_xlabel("Silhouette coefficient"); ax.set_ylabel("Clusters")
ax.legend(); plt.tight_layout(); plt.show()

## 14. PCA: 2D Visualisation of Customer Segments

In [ ]:
pca_full = PCA().fit(X_scaled)
cum = np.cumsum(pca_full.explained_variance_ratio_)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
pcs = pca.fit_transform(X_scaled)
cust["PC1"], cust["PC2"] = pcs[:, 0], pcs[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
axes[0].bar(range(1, len(cum) + 1), pca_full.explained_variance_ratio_, alpha=0.6, label="Individual")
axes[0].step(range(1, len(cum) + 1), cum, where="mid", color="red", label="Cumulative")
axes[0].set_title("PCA Explained Variance"); axes[0].set_xlabel("Principal component"); axes[0].legend()

sns.scatterplot(data=cust, x="PC1", y="PC2", hue="Cluster", palette="tab10", s=18, alpha=0.7, ax=axes[1])
axes[1].set_title(f"K-Means Segments in 2D PCA Space\n(PC1 {pca.explained_variance_ratio_[0]*100:.1f}%, PC2 {pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.tight_layout(); plt.show()

print(f"Variance captured by 2 components: {cum[1]*100:.1f}%")
display(pd.DataFrame(pca.components_.T, index=CLUSTER_FEATURES, columns=["PC1", "PC2"]).round(3))

## 15. Cluster-wise Customer Profiling
Each cluster is summarised in original units (GBP, days, units) and named using its position relative to the overall average (standardised cluster centres). The **high-value cluster** is the one with the highest average Monetary value.

In [ ]:
profile = cust.groupby("Cluster")[PROFILE_FEATURES].mean()
profile.insert(0, "Customers", cust["Cluster"].value_counts().sort_index())
profile.insert(1, "% of base", (profile["Customers"] / len(cust) * 100).round(1))
profile["Revenue share %"] = (cust.groupby("Cluster")["Monetary"].sum() / cust["Monetary"].sum() * 100).round(1)
display(profile.round(2))

# standardised centres (z-scores) to drive naming
centres_z = pd.DataFrame(kmeans.cluster_centers_, columns=CLUSTER_FEATURES)

HV_CLUSTER = int(profile["Monetary"].idxmax())

def name_segment(c):
    R, F, M = centres_z.loc[c, "Recency"], centres_z.loc[c, "Frequency"], centres_z.loc[c, "Monetary"]
    if c == HV_CLUSTER:                 return "Champions (High-Value)"
    if R > 0.5 and F < 0:               return "Lost / Dormant"
    if R > 0.5:                         return "At-Risk"
    if F > 0.2 or M > 0.2:              return "Loyal Regulars"
    return "New / Low-Engagement"

seg_type = {c: name_segment(c) for c in range(K)}
seg_label = {c: f"C{c}: {seg_type[c]}" for c in range(K)}
cust["SegmentType"] = cust["Cluster"].map(seg_type)
cust["Segment"] = cust["Cluster"].map(seg_label)

profile_named = profile.copy(); profile_named.insert(0, "Segment", [seg_label[c] for c in profile.index])
display(profile_named.round(2).set_index("Segment"))

plt.figure(figsize=(9, 0.9 * K + 2))
sns.heatmap(centres_z.rename(index=seg_label), annot=True, fmt=".2f", cmap="RdYlGn_r", center=0)
plt.title("Standardised cluster centres (z-scores; Recency: lower is better)"); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, col in zip(axes, ["Recency", "Frequency", "Monetary"]):
    sns.boxplot(data=cust, x="Cluster", y=np.log1p(cust[col]), palette="tab10", ax=ax)
    ax.set_ylabel(f"log1p({col})"); ax.set_title(f"{col} by cluster")
plt.tight_layout(); plt.show()

print(f"High-value cluster -> {seg_label[HV_CLUSTER]}")

### 15.1 Interpretation of the Segments
* **Champions (High-Value)**: recent, frequent buyers with the highest spend and quantity. They are usually a small share of customers but contribute a disproportionately large share of revenue (see the *Revenue share %* column).
* **Loyal Regulars**: moderate-to-good frequency and spend, reasonably recent activity. This is the group most likely to be upgraded to Champions.
* **At-Risk**: customers who used to buy but have not purchased for a long time.
* **Lost / Dormant**: long inactivity and low frequency and spend.
* **New / Low-Engagement**: recent, low-spend and low-frequency buyers, mostly new or occasional customers.

Which of these labels appear depends on the clusters found with the chosen k; the table above is the source of truth.

## 16. Predicting High-Value Customers (Supervised Learning)
**Target:** `HighValue = 1` if the customer belongs to the high-value cluster identified above, otherwise `0`.

**Features:** the engineered customer features (log-transformed and capped). Random Forest and an RBF SVM are trained on a stratified 80/20 split, with `class_weight="balanced"` to handle class imbalance and cross-validated hyper-parameter tuning.

> **Note on label leakage:** the target comes from clustering on these same features, so high accuracy is expected. The models essentially learn to reproduce the cluster boundary. A reduced-feature check is included in Section 19 to show that the result does not depend on a single column.

In [ ]:
y = (cust["Cluster"] == HV_CLUSTER).astype(int)
X = X_proc.copy()

print("Class balance (1 = high-value):")
print(y.value_counts().rename({0: "Other customers", 1: "High-value"}).to_string())
print(f"High-value share: {y.mean()*100:.2f}%")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("\nTrain:", X_train.shape, "Test:", X_test.shape)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# ---- Random Forest
rf_grid = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    param_grid={"n_estimators": [200, 400], "max_depth": [None, 8], "min_samples_leaf": [1, 3]},
    scoring="f1", cv=cv, n_jobs=-1).fit(X_train, y_train)
rf = rf_grid.best_estimator_
print("\nBest RF params :", rf_grid.best_params_)

# ---- SVM (scaling inside the pipeline so no information leaks from the test set)
svm_pipe = Pipeline([("scale", StandardScaler()),
                     ("svc", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE))])
svm_grid = GridSearchCV(svm_pipe, param_grid={"svc__C": [0.5, 1, 5, 10], "svc__gamma": ["scale", 0.05, 0.2]},
                        scoring="f1", cv=cv, n_jobs=-1).fit(X_train, y_train)
svm = svm_grid.best_estimator_
print("Best SVM params:", svm_grid.best_params_)

## 17. Model Evaluation and Comparison

In [ ]:
def evaluate(name, model):
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    return {"Model": name,
            "Accuracy": accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred, zero_division=0),
            "Recall": recall_score(y_test, pred, zero_division=0),
            "F1-Score": f1_score(y_test, pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, proba)}

results = pd.DataFrame([evaluate("Random Forest", rf), evaluate("SVM (RBF)", svm)]).set_index("Model")
display(results.round(4))

cv_f1 = {n: cross_val_score(m, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE), scoring="f1").mean()
         for n, m in [("Random Forest", rf), ("SVM (RBF)", svm)]}
print("5-fold CV F1 on training data:", {k: round(v, 4) for k, v in cv_f1.items()})

for name, m in [("Random Forest", rf), ("SVM (RBF)", svm)]:
    print(f"\n=== {name}: classification report ===")
    print(classification_report(y_test, m.predict(X_test), target_names=["Other", "High-Value"], zero_division=0))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, (name, m) in zip(axes[:2], [("Random Forest", rf), ("SVM (RBF)", svm)]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, m.predict(X_test)), display_labels=["Other", "High-Value"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion Matrix: {name}")
for name, m in [("Random Forest", rf), ("SVM (RBF)", svm)]:
    fpr, tpr, _ = roc_curve(y_test, m.predict_proba(X_test)[:, 1])
    axes[2].plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]):.3f})")
axes[2].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[2].set_xlabel("False Positive Rate"); axes[2].set_ylabel("True Positive Rate"); axes[2].set_title("ROC Curves"); axes[2].legend()
plt.tight_layout(); plt.show()

results.plot(kind="bar", figsize=(9, 4), ylim=(0, 1.05), rot=0, colormap="viridis")
plt.title("Model Performance Comparison"); plt.legend(loc="lower right", ncol=3); plt.tight_layout(); plt.show()

## 18. Feature Importance Analysis

In [ ]:
imp_rf = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
perm = permutation_importance(rf, X_test, y_test, n_repeats=15, scoring="f1", random_state=RANDOM_STATE, n_jobs=-1)
imp_perm = pd.Series(perm.importances_mean, index=X.columns).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
imp_rf.plot(kind="barh", ax=axes[0], color="teal"); axes[0].set_title("Random Forest: Gini Importance")
imp_perm.plot(kind="barh", ax=axes[1], xerr=pd.Series(perm.importances_std, index=X.columns).reindex(imp_perm.index), color="indianred")
axes[1].set_title("Permutation Importance (drop in F1 on test set)")
plt.tight_layout(); plt.show()

display(pd.DataFrame({"Gini importance": imp_rf, "Permutation importance": imp_perm}).sort_values("Gini importance", ascending=False).round(4))

## 19. Robustness Check: Predicting Without Monetary and TotalQuantity

In [ ]:
reduced_cols = [c for c in X.columns if c not in ("Monetary", "TotalQuantity")]
Xtr_r, Xte_r = X_train[reduced_cols], X_test[reduced_cols]
rf_r = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr_r, y_train)
pr = rf_r.predict(Xte_r)
print("Random Forest without Monetary/TotalQuantity:")
print(f"  Accuracy={accuracy_score(y_test, pr):.4f}  Precision={precision_score(y_test, pr, zero_division=0):.4f}  "
      f"Recall={recall_score(y_test, pr, zero_division=0):.4f}  F1={f1_score(y_test, pr, zero_division=0):.4f}  "
      f"ROC-AUC={roc_auc_score(y_test, rf_r.predict_proba(Xte_r)[:, 1]):.4f}")

## 20. Interactive 3D Visualisation of Customer Clusters (Plotly)
Axes are `log1p(Recency)`, `log1p(Frequency)` and `log1p(Monetary)`. Rotate, zoom and hover over points in Colab. Because interactive plots do not survive PDF export, a static matplotlib version follows for the PDF submission.

In [ ]:
plot_df = cust.reset_index()[["CustomerID", "Segment", "Recency", "Frequency", "Monetary"]].copy()
for c in ["Recency", "Frequency", "Monetary"]:
    plot_df[f"log_{c}"] = np.log1p(plot_df[c])

fig3d = px.scatter_3d(plot_df, x="log_Recency", y="log_Frequency", z="log_Monetary", color="Segment",
                      hover_data={"CustomerID": True, "Recency": True, "Frequency": True, "Monetary": ":.2f",
                                  "log_Recency": False, "log_Frequency": False, "log_Monetary": False},
                      opacity=0.75, title=f"3D RFM Customer Clusters ({STUDENT_NAME}, {ROLL_NO})")
fig3d.update_traces(marker=dict(size=3))
fig3d.update_layout(scene=dict(xaxis_title="log Recency (days)", yaxis_title="log Frequency (orders)", zaxis_title="log Monetary (GBP)"),
                    legend_title="Segment", height=650)
fig3d.write_html("customer_clusters_3d.html")
fig3d.show()

In [ ]:
# Static 3D view (appears in PDF export)
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
for seg, d in plot_df.groupby("Segment"):
    ax.scatter(d["log_Recency"], d["log_Frequency"], d["log_Monetary"], s=8, alpha=0.6, label=seg)
ax.set_xlabel("log Recency"); ax.set_ylabel("log Frequency"); ax.set_zlabel("log Monetary")
ax.set_title("3D RFM Customer Clusters (static)"); ax.legend(loc="upper left", fontsize=8); ax.view_init(elev=22, azim=-55)
plt.tight_layout(); plt.show()

## 21. Segment-wise Targeted Marketing Recommendations

In [ ]:
playbook = {
    "Champions (High-Value)": {
        "Goal": "Retain and grow lifetime value",
        "Strategy": "VIP / loyalty tier, early access to new products, exclusive bundles, personal account-manager style emails, referral rewards. Avoid heavy discounting; they already buy."},
    "Loyal Regulars": {
        "Goal": "Upgrade towards Champions",
        "Strategy": "Cross-sell and up-sell based on past purchases, points-based loyalty programme, free-shipping thresholds just above their average basket, subscription or replenishment reminders."},
    "At-Risk": {
        "Goal": "Win back before they churn",
        "Strategy": "Personalised 'we miss you' campaigns, time-limited comeback offers, reminders of previously purchased items, short feedback survey to learn why activity dropped."},
    "Lost / Dormant": {
        "Goal": "Low-cost reactivation or sunset",
        "Strategy": "One or two low-cost reactivation emails with a strong incentive; if no response, suppress from paid campaigns to save marketing spend."},
    "New / Low-Engagement": {
        "Goal": "Build the second purchase and habit",
        "Strategy": "Welcome series, onboarding content, first-repeat-purchase discount, product recommendations from best-sellers, low-minimum bundles."},
}

rows = []
for c in range(K):
    t = seg_type[c]
    p = profile.loc[c]
    rows.append({"Segment": seg_label[c], "Customers": int(p["Customers"]), "% of base": p["% of base"],
                 "Revenue share %": p["Revenue share %"], "Avg Recency (d)": round(p["Recency"], 0),
                 "Avg Orders": round(p["Frequency"], 1), "Avg Spend (GBP)": round(p["Monetary"], 0),
                 "Goal": playbook[t]["Goal"], "Recommended strategy": playbook[t]["Strategy"]})
reco = pd.DataFrame(rows).set_index("Segment")
pd.set_option("display.max_colwidth", None)
display(reco)
reco.to_csv("segment_marketing_recommendations.csv")

## 22. Results Summary and Interpretation

In [ ]:
best = results["F1-Score"].idxmax()
print(f"Student: {STUDENT_NAME} | Roll No: {ROLL_NO}\n")
print(f"Customers analysed         : {len(cust):,}")
print(f"Clusters (k)               : {K}  (Elbow + Silhouette)")
print(f"K-Means Silhouette         : {sil_km:.4f}")
print(f"Hierarchical Silhouette    : {sil_hc:.4f}   (ARI vs K-Means = {ari:.3f})")
print(f"PCA variance in 2D         : {cum[1]*100:.1f}%")
hv = profile.loc[HV_CLUSTER]
print(f"High-value segment         : {seg_label[HV_CLUSTER]} -> {hv['% of base']}% of customers, {hv['Revenue share %']}% of revenue")
print(f"Best classifier (F1)       : {best}")
print(f"Top predictive features    : {', '.join(imp_rf.sort_values(ascending=False).index[:3])}")

## 23. Conclusion
* The raw transaction log was cleaned (missing customers, cancellations, invalid quantities and prices, duplicates) and aggregated into customer-level RFM and behavioural features. Log transformation, winsorisation and standardisation made the heavy-tailed data suitable for distance-based algorithms.
* The Elbow Method together with Silhouette Scores guided the choice of k. K-Means and Ward hierarchical clustering were compared using Silhouette Score and the Adjusted Rand Index, and the dendrogram confirmed the hierarchical structure behind the chosen k.
* PCA showed that two components capture most of the variance, giving a clear 2D view of the segments. The 3D Plotly chart shows the separation in Recency, Frequency and Monetary space.
* A small high-value segment generates a disproportionate share of revenue, so retaining it is the top priority. The other segments need different treatment: upgrade (Loyal Regulars), win back (At-Risk), cheap reactivation or suppression (Lost), and onboarding (New).
* Random Forest and SVM both predict high-value membership with high accuracy, Precision, Recall, F1 and ROC-AUC. Because the label is derived from the same features, these scores partly reflect how well the models recover the cluster boundary, which Section 19 checks with a reduced feature set. In practice the classifier can score new customers as they arrive, without re-running clustering.
* **Limitations:** clusters depend on the chosen features and k, the data covers a single retailer over one year, and about a quarter of rows had no CustomerID and were excluded.

---
*Prepared by **Tanisha Sharma** (Roll No. **23102C0077**).*